# SHViT + FGVC Aircraft on Google Colab

This notebook walks through:
1. Enabling GPU and checking the environment
2. Cloning SHViT and installing dependencies
3. Downloading pretrained SHViT-S4 weights
4. Downloading FGVC Aircraft with Tip-Adapter style preprocessing
5. Verifying the model loads and runs inference
6. Running SHViT's official eval script

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`
> All outputs from this stage are saved under `/content/CV_Research_Paper_FGVCAircraft/`.


## 0. Check GPU & environment

In [11]:
import torch

print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

import sys
print('Python version  :', sys.version.split()[0])

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM            : 102.0 GB
Python version  : 3.12.13


## 1. (Optional) Mount Google Drive

FGVC Aircraft is ~3 GB (10,200 images). Mounting Drive lets you keep the
unpacked dataset and downloaded weights between Colab restarts. Skip this
cell if you are happy to re-download every session.


In [12]:
USE_DRIVE = False   # set True to persist data + outputs in Google Drive

OUT_ROOT = '/content/CV_Research_Paper_FGVCAircraft'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/fgvc_aircraft_data'
    OUT_ROOT  = '/content/drive/MyDrive/CV_Research_Paper_FGVCAircraft'
else:
    DATA_ROOT = '/content/fgvc_aircraft_data'

import os
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print('Dataset will be stored at:', DATA_ROOT)
print('Outputs will be written under:', OUT_ROOT)


Dataset will be stored at: /content/fgvc_aircraft_data
Outputs will be written under: /content/CV_Research_Paper_FGVCAircraft


## 2. Clone SHViT

In [13]:
import os

if not os.path.isdir('/content/SHViT'):
    !git clone https://github.com/ysj9909/SHViT.git /content/SHViT
else:
    print('SHViT already cloned, skipping.')

!ls /content/SHViT

SHViT already cloned, skipping.
acc_vs_thro.png  engine.py	  losses.py  __pycache__       speed_test.py
data		 export_model.py  main.py    README.md	       utils.py
downstream	 LICENSE	  model      requirements.txt


## 3. Install dependencies

Colab ships with PyTorch 2.x which satisfies SHViT's `>=1.11` requirement,
so we only need to install the extra packages from `requirements.txt`.

`--no-deps` on timm avoids overwriting Colab's torch/torchvision with
the older versions timm 0.5.4 would otherwise pull in.

In [14]:
# scikit-image==0.19.3 from SHViT's requirements has no wheels for Python
# 3.12 (Colab's default) — and we don't actually need it. Install only what
# the SHViT model architecture needs.
!pip install -q timm==0.5.4 --no-deps
!pip install -q einops==0.4.1 easydict
print('Dependencies installed.')

Dependencies installed.


## 4. Download SHViT-S4 pretrained weights

In [15]:
WEIGHTS_DIR = '/content/weights'
WEIGHTS_PATH = f'{WEIGHTS_DIR}/shvit_s4.pth'

os.makedirs(WEIGHTS_DIR, exist_ok=True)

if not os.path.exists(WEIGHTS_PATH):
    !wget -q --show-progress \
        https://github.com/ysj9909/SHViT/releases/download/v1.0/shvit_s4.pth \
        -O {WEIGHTS_PATH}
else:
    print('Weights already downloaded, skipping.')

size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
print(f'Checkpoint size: {size_mb:.1f} MB  ->  {WEIGHTS_PATH}')

Weights already downloaded, skipping.
Checkpoint size: 266.7 MB  ->  /content/weights/shvit_s4.pth


## 5. Download FGVC Aircraft with Tip-Adapter style preprocessing

This clones the FGVC Aircraft project and runs `prepare_fgvc_aircraft.py`,
which downloads the dataset via `torchvision.datasets.FGVCAircraft` and
re-layouts it into the Tip-Adapter folder convention. FGVC Aircraft ships
its own canonical train/val/test split files
(`images_variant_{train,val,test}.txt`), so no CoOp JSON download is involved.


In [16]:
import os, shutil

REPO_DIR = '/content/Vision_Project_spring_26'
if not os.path.isdir(REPO_DIR):
    !git clone -b Vision_Project_spring_26_FGVCAircraft \
        https://github.com/saif-farid-tech/Vision_Project_spring_26.git {REPO_DIR}

# Make sure the prepare script + dataset helpers are importable from /content
for fname in [
    'prepare_fgvc_aircraft.py',
    'splits.py',
    'metrics.py',
    'augmentation.py',
]:
    shutil.copy(f'{REPO_DIR}/{fname}', f'/content/{fname}')

# Vendor the datasets/ package (Tip-Adapter utilities)
DATASETS_DST = '/content/datasets'
if os.path.isdir(DATASETS_DST):
    shutil.rmtree(DATASETS_DST)
shutil.copytree(f'{REPO_DIR}/datasets', DATASETS_DST)

!pip install -q gdown
!python /content/prepare_fgvc_aircraft.py --root {DATA_ROOT}


[fgvc_aircraft] dataset already present: /content/fgvc_aircraft_data/fgvc_aircraft

Done. Dataset root: /content/fgvc_aircraft_data/fgvc_aircraft
  classes : 100
  train   : 3,334
  val     : 3,333
  test    : 3,333


In [17]:
# Sanity check
from pathlib import Path
ds_root = Path(DATA_ROOT) / 'fgvc_aircraft'
img_dir = ds_root / 'images'
variants_path = ds_root / 'variants.txt'
train_split = ds_root / 'images_variant_train.txt'

n_img = sum(1 for _ in img_dir.glob('*.jpg')) if img_dir.exists() else 0
n_variants = sum(1 for _ in open(variants_path)) if variants_path.exists() else 0
print(f'On-disk:  {n_img} images at {img_dir}')
print(f'Variants : {n_variants} classes in {variants_path}')
print(f'Train split file exists: {train_split.exists()}')


On-disk:  10000 images at /content/fgvc_aircraft_data/fgvc_aircraft/images
Variants : 100 classes in /content/fgvc_aircraft_data/fgvc_aircraft/variants.txt
Train split file exists: True


## 6. Verify model loads and runs inference

Loads the SHViT-S4 checkpoint and runs 50 FGVC Aircraft images through it.
Predictions are ImageNet class indices (not FGVC Aircraft labels) — accuracy
will be ~zero until the model is fine-tuned. The goal here is just to
confirm no import / shape errors occur.


In [18]:
import sys, time, pathlib
import torch
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

sys.path.insert(0, '/content/SHViT')

from model import shvit
import timm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

model_shvit = timm.create_model('shvit_s4', pretrained=False, num_classes=1000)

ckpt = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=False)
state_dict = ckpt.get('model', ckpt)
missing, unexpected = model_shvit.load_state_dict(state_dict, strict=False)
print(f'Missing keys: {len(missing)}   Unexpected keys: {len(unexpected)}')

model_shvit.to(DEVICE).eval()
print('Model loaded successfully.')


Using device: cuda
Missing keys: 0   Unexpected keys: 0
Model loaded successfully.


In [19]:
NUM_IMAGES = 50

# CLIP / Tip-Adapter normalization
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

# FGVC Aircraft filenames are numeric image IDs (e.g. 0034309.jpg).
# The variant for each id lives in the split files, not the filename, so
# we read images_variant_*.txt to recover it.
ds_root = pathlib.Path(DATA_ROOT) / 'fgvc_aircraft'
image_dir = ds_root / 'images'

id_to_variant = {}
for fname in ('images_variant_train.txt',
              'images_variant_val.txt',
              'images_variant_test.txt'):
    path = ds_root / fname
    if not path.exists():
        continue
    with open(path) as f:
        for line in f:
            parts = line.strip().split(' ')
            if len(parts) >= 2:
                id_to_variant[parts[0]] = ' '.join(parts[1:])

items = []
for img_path in sorted(image_dir.glob('*.jpg'))[:NUM_IMAGES]:
    variant = id_to_variant.get(img_path.stem, 'unknown')
    items.append((img_path, variant))

print(f'Running inference on {len(items)} images ...')
t0 = time.perf_counter()
results = []
with torch.no_grad():
    for img_path, true_class in items:
        x = transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
        pred = int(model_shvit(x).argmax(1).item())
        results.append((img_path.name, true_class, pred))

elapsed = time.perf_counter() - t0
print(f'\n{"Image":<30} {"True class":<25} {"Pred idx":>8}')
print('-' * 65)
for name, cls, pred in results[:15]:
    print(f'{name:<30} {cls:<25} {pred:>8}')
print(f'\nTotal: {elapsed:.2f}s  ({elapsed/len(results)*1000:.1f} ms/image)')
print('\n[OK] Model ran without errors.')


Running inference on 50 images ...

Image                          True class                Pred idx
-----------------------------------------------------------------
0034309.jpg                    DC-8                           805
0034958.jpg                    737-200                        404
0037511.jpg                    DC-9-30                        895
0037512.jpg                    737-200                        404
0038598.jpg                    MD-11                          908
0038626.jpg                    DC-9-30                        908
0038671.jpg                    Boeing 717                     404
0041419.jpg                    Gulfstream IV                  404
0043750.jpg                    C-47                           895
0043890.jpg                    DC-10                          404
0043892.jpg                    Challenger 600                 404
0045128.jpg                    747-200                        404
0045133.jpg                    DC-3     

## 7. Run SHViT's official eval script

FGVC Aircraft stores every image in one flat `images/` folder, so we first
build a temporary per-class symlink tree from the canonical
`images_variant_{train,val}.txt` split files; SHViT's `--data-set IMNET`
then accepts it as an ImageFolder-style dataset. Expect ~zero accuracy
relative to ImageNet-1K classes — fine-tuning happens in Stage 3.


In [20]:
import re, os, shutil

with open('/content/SHViT/main.py', 'r') as f:
    content = f.read()
content = re.sub(r"torch\.load\(([^,]+),\s*map_location='cpu'\)",
                 r"torch.load(\1, map_location='cpu', weights_only=False)",
                 content)
with open('/content/SHViT/main.py', 'w') as f:
    f.write(content)

# FGVC Aircraft has flat images/ + canonical split files (no CoOp JSON).
# Build a per-class symlink tree from the train + val split files so that
# SHViT's IMNET data-set finds an ImageFolder-style layout.
IM_ROOT = '/content/imnet_fgvc_aircraft_sanity'
DS_ROOT = f'{DATA_ROOT}/fgvc_aircraft'

if os.path.isdir(IM_ROOT):
    shutil.rmtree(IM_ROOT)
for tgt, split_fname in (('train', 'images_variant_train.txt'),
                         ('val',   'images_variant_val.txt')):
    with open(f'{DS_ROOT}/{split_fname}') as f:
        for line in f:
            parts = line.strip().split(' ')
            if len(parts) < 2:
                continue
            imid = parts[0]
            variant = ' '.join(parts[1:])
            cls_safe = variant.replace(' ', '_').replace('/', '_')
            cls_dir = f'{IM_ROOT}/{tgt}/{cls_safe}'
            os.makedirs(cls_dir, exist_ok=True)
            src = f'{DS_ROOT}/images/{imid}.jpg'
            dst = f'{cls_dir}/{imid}.jpg'
            if not os.path.exists(dst):
                os.symlink(src, dst)

!python /content/SHViT/main.py \
    --model shvit_s4 \
    --eval \
    --resume {WEIGHTS_PATH} \
    --data-path {IM_ROOT} \
    --data-set IMNET \
    --batch-size 64 \
    --num_workers 2 \
    --device cuda


Not using distributed mode
Creating model: shvit_s4
number of params: 16588484
/usr/local/lib/python3.12/dist-packages/timm/utils/cuda.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler()
Loading local checkpoint at /content/weights/shvit_s4.pth
<All keys matched successfully>
Evaluating model: shvit_s4
/content/SHViT/engine.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Test:  [ 0/35]  eta: 0:04:39  loss: 8.0036 (8.0036)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0000)  time: 7.9808  data: 0.7295  max mem: 479
Test:  [10/35]  eta: 0:00:23  loss: 8.2456 (8.2365)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0000)  time: 0.9404  data: 0.2740  max mem: 479
Test:  [20/35]  eta: 0:00:10  loss: 8.2811 (8.3116)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0000)  time

## Next steps — fine-tuning on FGVC Aircraft

To actually train SHViT on FGVC Aircraft, head over to Stage 3 and run
`finetune_shvit_fgvc_aircraft.py`, which uses the Tip-Adapter split JSON,
CLIP normalization, RandAugment / RandomErasing / Mixup / CutMix /
label-smoothing, AGC-style gradient clipping, and cosine LR with warmup
(the same recipe as the SHViT paper, but with `--nb_classes 100`).
